# ApexPlanet Internship — Task 2: SQL for Data Extraction & Analysis

**Dataset:** UCI Online Retail  
**Environment:** Google Colab + SQLite + SQLAlchemy + Pandas

This notebook covers SQL fundamentals, advanced SQL, Python/SQL integration, and 10 business questions.

In [ ]:
!pip -q install pandas openpyxl sqlalchemy

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import os

file_name = next(iter(uploaded))
df = pd.read_excel(file_name)
print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
display(df.isnull().sum().to_frame("missing_values"))
print("Negative quantities:", (df["Quantity"] < 0).sum())
print("Zero prices:", (df["UnitPrice"] == 0).sum())
print("Cancelled invoices:", df["InvoiceNo"].astype(str).str.startswith("C").sum())

## Data cleaning

For the main sales-analysis table:
- remove cancelled invoices
- remove missing CustomerID
- remove non-positive quantities/prices
- remove exact duplicates
- convert InvoiceDate to datetime
- create `TotalSales = Quantity × UnitPrice`

In [ ]:
clean = df.copy()
clean = clean.drop_duplicates()
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)
clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[clean["Quantity"] > 0]
clean = clean[clean["UnitPrice"] > 0]
clean = clean.dropna(subset=["CustomerID"])
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"])
clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["TotalSales"] = clean["Quantity"] * clean["UnitPrice"]

print("Cleaned shape:", clean.shape)
display(clean.head())

In [ ]:
os.makedirs("processed", exist_ok=True)
clean.to_csv("processed/online_retail_clean.csv", index=False)
print("Saved processed/online_retail_clean.csv")

## Create SQLite database

In [ ]:
from sqlalchemy import create_engine, text
engine = create_engine("sqlite:///online_retail.db")
clean.to_sql("transactions", engine, if_exists="replace", index=False)
print("SQLite database created.")

In [ ]:
def run_sql(query):
    return pd.read_sql(text(query), engine)

run_sql('''
SELECT *
FROM transactions
LIMIT 10;
''')

In [ ]:
run_sql('''
SELECT InvoiceNo, Description, Quantity, UnitPrice, TotalSales
FROM transactions
WHERE Country = 'United Kingdom'
  AND TotalSales > 100
LIMIT 20;
''')

In [ ]:
run_sql('''
SELECT InvoiceNo, Description, TotalSales
FROM transactions
ORDER BY TotalSales DESC
LIMIT 5;
''')

In [ ]:
run_sql('''
SELECT Country, ROUND(SUM(TotalSales), 2) AS TotalSales
FROM transactions
GROUP BY Country
ORDER BY TotalSales DESC;
''')

In [ ]:
run_sql('''
SELECT Country, ROUND(SUM(TotalSales), 2) AS TotalSales
FROM transactions
GROUP BY Country
HAVING SUM(TotalSales) > 100000
ORDER BY TotalSales DESC;
''')

## JOIN practice

In [ ]:
customers = clean[["CustomerID", "Country"]].drop_duplicates()
products = clean[["StockCode", "Description"]].drop_duplicates(subset=["StockCode"])
customers.to_sql("customers", engine, if_exists="replace", index=False)
products.to_sql("products", engine, if_exists="replace", index=False)
print("Created customers, products, transactions")

In [ ]:
run_sql('''
SELECT t.InvoiceNo, t.StockCode, p.Description, t.Quantity, t.TotalSales
FROM transactions t
INNER JOIN products p ON t.StockCode = p.StockCode
LIMIT 20;
''')

In [ ]:
run_sql('''
SELECT c.CustomerID, c.Country, ROUND(SUM(t.TotalSales), 2) AS CustomerSales
FROM customers c
INNER JOIN transactions t ON c.CustomerID = t.CustomerID
GROUP BY c.CustomerID, c.Country
ORDER BY CustomerSales DESC
LIMIT 10;
''')

## Advanced SQL: subqueries, CTEs, window functions and views

In [ ]:
run_sql('''
SELECT CustomerID, ROUND(SUM(TotalSales), 2) AS CustomerSales
FROM transactions
GROUP BY CustomerID
HAVING SUM(TotalSales) >
       (SELECT AVG(CustomerTotal)
        FROM (
            SELECT SUM(TotalSales) AS CustomerTotal
            FROM transactions
            GROUP BY CustomerID
        ));
''')

In [ ]:
run_sql('''
WITH monthly_sales AS (
    SELECT strftime('%Y-%m', InvoiceDate) AS Month,
           SUM(TotalSales) AS Sales
    FROM transactions
    GROUP BY strftime('%Y-%m', InvoiceDate)
)
SELECT Month, ROUND(Sales, 2) AS Sales
FROM monthly_sales
ORDER BY Month;
''')

In [ ]:
run_sql('''
WITH customer_sales AS (
    SELECT CustomerID, SUM(TotalSales) AS Sales
    FROM transactions
    GROUP BY CustomerID
)
SELECT CustomerID, ROUND(Sales, 2) AS Sales,
       ROW_NUMBER() OVER (ORDER BY Sales DESC) AS RowNum
FROM customer_sales
LIMIT 10;
''')

In [ ]:
run_sql('''
WITH product_sales AS (
    SELECT StockCode, MAX(Description) AS Description,
           SUM(TotalSales) AS Sales
    FROM transactions
    GROUP BY StockCode
)
SELECT StockCode, Description, ROUND(Sales, 2) AS Sales,
       RANK() OVER (ORDER BY Sales DESC) AS SalesRank
FROM product_sales
LIMIT 10;
''')

In [ ]:
run_sql('''
WITH monthly_sales AS (
    SELECT strftime('%Y-%m', InvoiceDate) AS Month,
           SUM(TotalSales) AS Sales
    FROM transactions
    GROUP BY strftime('%Y-%m', InvoiceDate)
)
SELECT Month, ROUND(Sales, 2) AS Sales,
       ROUND(LAG(Sales) OVER (ORDER BY Month), 2) AS PreviousMonthSales,
       ROUND((Sales - LAG(Sales) OVER (ORDER BY Month))
             / NULLIF(LAG(Sales) OVER (ORDER BY Month), 0) * 100, 2) AS MoM_Percent
FROM monthly_sales
ORDER BY Month;
''')

In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS customer_sales_summary"))
    conn.execute(text('''
        CREATE VIEW customer_sales_summary AS
        SELECT CustomerID,
               COUNT(DISTINCT InvoiceNo) AS Orders,
               SUM(Quantity) AS Units,
               ROUND(SUM(TotalSales), 2) AS Sales
        FROM transactions
        GROUP BY CustomerID
    '''))

run_sql('''
SELECT * FROM customer_sales_summary
ORDER BY Sales DESC LIMIT 10;
''')

# 10 Business Questions

In [ ]:
# 1. Top 5 products by sales
run_sql('''
SELECT StockCode, MAX(Description) AS Description,
       ROUND(SUM(TotalSales), 2) AS Sales
FROM transactions
GROUP BY StockCode
ORDER BY Sales DESC LIMIT 5;
''')

In [ ]:
# 2. Monthly sales trend
run_sql('''
SELECT strftime('%Y-%m', InvoiceDate) AS Month,
       ROUND(SUM(TotalSales), 2) AS Sales
FROM transactions
GROUP BY Month ORDER BY Month;
''')

In [ ]:
# 3. Top 10 customers by spend
run_sql('''
SELECT CustomerID, ROUND(SUM(TotalSales), 2) AS Spend
FROM transactions
GROUP BY CustomerID ORDER BY Spend DESC LIMIT 10;
''')

In [ ]:
# 4. Sales by country
run_sql('''
SELECT Country, ROUND(SUM(TotalSales), 2) AS Sales
FROM transactions
GROUP BY Country ORDER BY Sales DESC;
''')

In [ ]:
# 5. Average order value
run_sql('''
SELECT ROUND(SUM(TotalSales) / COUNT(DISTINCT InvoiceNo), 2) AS AverageOrderValue
FROM transactions;
''')

In [ ]:
# 6. Top products by quantity
run_sql('''
SELECT StockCode, MAX(Description) AS Description, SUM(Quantity) AS UnitsSold
FROM transactions
GROUP BY StockCode ORDER BY UnitsSold DESC LIMIT 10;
''')

In [ ]:
# 7. Repeat customers
run_sql('''
SELECT CustomerID, COUNT(DISTINCT InvoiceNo) AS Orders
FROM transactions
GROUP BY CustomerID HAVING Orders > 1
ORDER BY Orders DESC;
''')

In [ ]:
# 8. Products with highest average transaction value
run_sql('''
SELECT StockCode, MAX(Description) AS Description,
       ROUND(AVG(TotalSales), 2) AS AvgTransactionValue
FROM transactions
GROUP BY StockCode
HAVING COUNT(*) >= 10
ORDER BY AvgTransactionValue DESC LIMIT 10;
''')

In [ ]:
# 9. Country contribution to total sales
run_sql('''
WITH country_sales AS (
    SELECT Country, SUM(TotalSales) AS Sales
    FROM transactions GROUP BY Country
),
total AS (
    SELECT SUM(Sales) AS TotalSales FROM country_sales
)
SELECT Country, ROUND(Sales, 2) AS Sales,
       ROUND(Sales * 100.0 / TotalSales, 2) AS SalesSharePercent
FROM country_sales, total
ORDER BY Sales DESC;
''')

In [ ]:
# 10. Customers with declining consecutive-month sales
run_sql('''
WITH customer_month AS (
    SELECT CustomerID, strftime('%Y-%m', InvoiceDate) AS Month,
           SUM(TotalSales) AS Sales
    FROM transactions
    GROUP BY CustomerID, Month
),
comparison AS (
    SELECT CustomerID, Month, Sales,
           LAG(Sales) OVER (
               PARTITION BY CustomerID ORDER BY Month
           ) AS PreviousSales
    FROM customer_month
)
SELECT CustomerID, Month,
       ROUND(Sales, 2) AS Sales,
       ROUND(PreviousSales, 2) AS PreviousSales
FROM comparison
WHERE PreviousSales IS NOT NULL AND Sales < PreviousSales
ORDER BY (PreviousSales - Sales) DESC
LIMIT 20;
''')

In [ ]:
customer_summary = run_sql('''
SELECT * FROM customer_sales_summary ORDER BY Sales DESC;
''')
customer_summary.to_excel("task2_customer_sales_summary.xlsx", index=False)
print("Excel export created.")

## Task 2 checklist

- SQLite database
- Cleaning and derived sales metric
- SELECT / WHERE / ORDER BY / LIMIT
- GROUP BY / HAVING
- JOINs
- Subquery / CTE
- ROW_NUMBER / RANK / LAG
- View
- SQLAlchemy + pandas.read_sql
- 10 business questions
- Excel export

Next: save the notebook and add `.sql`, utility script, `requirements.txt`, and README to GitHub.